# S27 · Save your model, then write the web app that uses it

You have a trained model. Today you make it into something other people can use.

Today is a workshop with six steps. **This notebook is step 1.** It trains a small
model, saves it to a file, and loads it back. Then it writes the web app
(`app.py`) and the list of packages (`requirements.txt`). After that, you do
steps 2 to 6 outside the notebook. You run the app on your laptop and change it.
Then you put it on GitHub, deploy it, and update it. The text near the end of this notebook gives
each of those steps in order.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Run each cell from top to
  bottom (press the play button), and read the short note above each cell.
- New to the idea "a model is a function"? In short: numbers go in, and an answer
  comes out. Any new word is in `primers/glossary.md`.
- Never used a terminal or GitHub? That is fine. You need them only in steps 2, 4
  and 5, and those steps are written out one click at a time.
- Already confident with code? Look for the cells marked **Stretch (optional)**.

## How this notebook runs (please read)

A Streamlit app is a Python **script**. A web server runs it. It is not notebook
code. If we started that server inside this notebook, the notebook would wait
forever and never finish.

So we do not start the app here. This notebook only **trains, saves and loads**
the model. Then it **writes** the app to a file and **prints** it, so you can read
it. It never starts a server and never calls `streamlit run`. So it runs from top
to bottom, without internet, in a few seconds. You start the real app in step 2,
in a terminal.

## Setup

On **Google Colab**, run the next cell once. On **your own laptop** you already
installed everything with `uv`, so the next cell does nothing there.

In [ ]:
# This notebook uses only numpy, scikit-learn and joblib. Colab already has
# all three, so there is nothing to install here.
#
# Note: the web app also needs the "streamlit" library, but we do NOT import it
# in this notebook. Streamlit is needed only to RUN the app. On your laptop you
# install it in step 2. On the cloud, the cloud computer installs the packages
# in requirements.txt. So there is nothing to do now.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                                  # fast maths on rows of numbers
from sklearn.datasets import load_iris                # a tiny built-in dataset
from sklearn.model_selection import train_test_split  # to keep a fair test set aside
from sklearn.linear_model import LogisticRegression   # a simple, common classifier
from sklearn.metrics import accuracy_score            # to score the model honestly

## Step 1.1 · Load a small dataset that comes with scikit-learn

We need some data and a model. We want to spend today on deploying, so we use a
small dataset: the **iris** flowers. It comes inside scikit-learn. There is
nothing to download, and it runs on any computer.

Each flower has four measurements (the inputs). We want to guess which of three
species it is (the output). Putting each case into one of a few named groups is
called **classification**. A bank does the same when it puts loan applications
into approve, review or reject. A mandi does the same when it sorts mangoes into
Alphonso, Kesar and Totapuri by size and shape. Our problem is only smaller.

In [ ]:
# A seed so anything random is the same every time we run this.
np.random.seed(0)

# Load the dataset. X holds the measurements, y holds the species (0, 1 or 2).
iris = load_iris()
X = iris.data        # shape (150, 4): 150 flowers, 4 measurements each
y = iris.target      # shape (150,):   the species number for each flower

print("number of flowers      :", X.shape[0])
print("measurements per flower :", X.shape[1])
print("the four measurements   :", iris.feature_names)
print("the three species       :", list(iris.target_names))

## Step 1.2 · Look at the first few rows

Always look at your data before you build a model. Each row is one flower. The
four numbers are its measurements in centimetres. The arrow points to the true
species.

In [ ]:
# Print the measurements and the species for the first 5 flowers.
for flower_number in range(5):
    measurements = X[flower_number]
    species_number = y[flower_number]
    species_name = iris.target_names[species_number]
    print("flower", flower_number, ":", measurements, "->", species_name)

## Step 1.3 · Split the data into training data and test data

We train the model on one part of the data. We keep another part hidden. Then we
can check the model on flowers it has never seen. Checking the model only on data it
has already seen is not fair. It is like an exam with the same questions that the
students practised. A separate test set gives a fair check.

In [ ]:
# 80% of the flowers for training, 20% kept aside for a fair test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

print("training flowers:", X_train.shape[0])
print("test flowers    :", X_test.shape[0])

## Step 1.4 · Train a small model

We use **logistic regression**, a common classifier. The line `.fit(...)` is the
training step. The model looks at the training flowers and learns a rule that
gives a species for each set of measurements. This one line turns the data into
the function `f` that our app will use.

In [ ]:
# Create the model. max_iter gives it enough steps to finish learning.
model = LogisticRegression(max_iter=200)

# Train it on the TRAINING data only.
model.fit(X_train, y_train)

print("The model is trained.")

## Step 1.5 · Check the model on the hidden test flowers

Accuracy is the share of test flowers that the model labels correctly. We check it
so that we know the model is good enough to save and put online.

In [ ]:
# Ask the trained model to predict the species of the hidden test flowers.
test_predictions = model.predict(X_test)

# Compare the predictions with the true answers.
accuracy = accuracy_score(y_test, test_predictions)
print("accuracy on the hidden test flowers:", round(accuracy, 3))

## Step 1.6 · Save the trained model to a file (joblib)

This is the most important step of the day. Right now the trained `model` is only
in memory. When the notebook closes, it is lost. `joblib.dump` writes it to a
file on disk. We call this file the **saved model file**. Later, another program
(our app) can load this file on another day, with no training again. Writing an
object to a file like this is called **serializing** it.

In [ ]:
import joblib
import os

# The name of the saved model file. ".joblib" is a common ending;
# you will also often see ".pkl".
model_filename = "iris_model.joblib"

# Save (serialize) the trained model to that file. This is ONE line.
joblib.dump(model, model_filename)

print("Saved the trained model to:", model_filename)

# Show that the file is now on disk, and how small it is.
file_size_in_bytes = os.path.getsize(model_filename)
print("file size:", file_size_in_bytes, "bytes")

## Step 1.7 · Load the model back from the file

Now imagine that we are a new program, for example the web app. We do **not**
train anything again. We read the saved model file into a new variable with
`joblib.load`. This new variable is the same trained function.

In [ ]:
# Load (read back) the model from the file into a new variable.
loaded_model = joblib.load(model_filename)

print("Loaded the model back from:", model_filename)
print("type of the loaded object:", type(loaded_model).__name__)

## Step 1.8 · Predict with the LOADED model

Now we use the loaded model to guess the species of one new flower. The web app
does exactly this on every click. It loads the saved model once, and then calls it
on the numbers the user chose.

In [ ]:
# The four measurements of one new flower (sepal length, sepal width,
# petal length, petal width), in centimetres. The model needs a 2D array with
# one row per flower, so we put the four numbers inside another list.
new_flower = [[5.1, 3.5, 1.4, 0.2]]

# Use the LOADED model (not the original one) to predict.
predicted_species_number = loaded_model.predict(new_flower)[0]
predicted_species_name = iris.target_names[predicted_species_number]

print("measurements of the new flower:", new_flower[0])
print("predicted species number      :", predicted_species_number)
print("predicted species name        :", predicted_species_name)

## Step 1.9 · Check that the loaded model agrees with the original

Saving and loading must not change what the model does. We check that the loaded
model gives exactly the same predictions as the original model on the test
flowers.

In [ ]:
# Predictions from the original model and from the loaded model.
original_predictions = model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

# np.array_equal is True only if every prediction matches.
they_match = np.array_equal(original_predictions, loaded_predictions)

print("loaded model matches the original on every test flower:", they_match)

### Stretch (optional) · Show how sure the model is

New to code? You can skip this. A good app does not only say "setosa". It also
says how sure the model is. Most scikit-learn classifiers can give a
**probability** for each class with `predict_proba`. The app uses this to show a
confidence percentage next to the answer.

In [ ]:
# Probability the model gives to each of the three species, for our new flower.
probabilities = loaded_model.predict_proba(new_flower)[0]

for species_number in range(len(probabilities)):
    species_name = iris.target_names[species_number]
    percent = round(100 * probabilities[species_number], 1)
    print(species_name, ":", percent, "%")

# The confidence for the species that the model picked.
chosen_confidence = round(100 * probabilities[predicted_species_number], 1)
print()
print("The model picked", predicted_species_name,
      "with", chosen_confidence, "% confidence.")

## Step 1.10 · Write the web app (app.py)

Up to now the model lives in a notebook. Now we write the web app that will put it
on the internet. The web app is a Python script called `app.py`. This cell writes
it to disk for you, so this notebook makes the exact file that you will deploy.
You do not need to copy or type anything.

Remember: this cell only **writes** the app. It does not start it. This is on
purpose, so that the notebook does not stop and wait for a server.

In [ ]:
# The full text of our Streamlit web app, kept as one block of text.
# We write it to app.py, so this notebook makes the exact file that you
# deploy. You do not copy anything by hand, so there are no typing mistakes.
#
# We only WRITE the file here. We do NOT run it. A web server inside a
# notebook would never finish. You run the app in step 2, in a terminal.
app_source = r'''# app.py - a small Streamlit web app that uses a trained model.
#
# To run it on your laptop, open a terminal in this folder and type:
#     streamlit run app.py
# If "streamlit" is not found, type this instead:
#     python -m streamlit run app.py
#
# How Streamlit works: every time the user changes an input, this whole file
# runs again, from the first line to the last. So read it from top to bottom.
#
# IMPORTANT: this is a Python SCRIPT, not a notebook. Do not run it cell by
# cell, and do not run it inside a notebook. Run it in a terminal with the
# command shown above. (The session notebook writes this same file for you,
# so you do not need to copy it by hand.)

import streamlit as st          # the web-app library (the only new tool today)
import numpy as np              # to make the row of input numbers
import joblib                   # to load the saved model file from the notebook
from sklearn.datasets import load_iris   # only for the species names below


# --- Block 1: load the saved model (only once) ----------------------------- #
# @st.cache_resource tells Streamlit: run this function only ONCE, then keep
# the result in memory and use it again on every later run. Without it, the
# file would be loaded again on every click, and the app would be slow.
@st.cache_resource
def load_model():
    # Read the saved model file that the notebook made with joblib.dump.
    # The file is in the same folder as this script (and in the same repo),
    # so the file name alone is enough.
    return joblib.load("iris_model.joblib")


# Load the model, and the names of the three species.
model = load_model()
species_names = load_iris().target_names   # setosa, versicolor, virginica


# --- Block 2: the page heading --------------------------------------------- #
st.title("Iris flower classifier")
st.write(
    "Enter the four measurements of a flower (in centimetres). The model "
    "will guess its species. The model was trained once and saved. This app "
    "only loads the saved model and predicts."
)


# --- Block 3: collect the inputs from the user ----------------------------- #
# Each slider is one line. It gives back the number that the user chose.
# The sliders are in the sidebar on the left, so the page stays clean.
st.sidebar.header("Flower measurements")
sepal_length = st.sidebar.slider("Sepal length (cm)", 4.0, 8.0, 5.1)
sepal_width = st.sidebar.slider("Sepal width (cm)", 2.0, 4.5, 3.5)
petal_length = st.sidebar.slider("Petal length (cm)", 1.0, 7.0, 1.4)
petal_width = st.sidebar.slider("Petal width (cm)", 0.1, 2.5, 0.2)

# Show the inputs again, so the user can see the numbers the model used.
st.subheader("Your inputs")
st.write("Sepal length:", sepal_length, "cm")
st.write("Sepal width :", sepal_width, "cm")
st.write("Petal length:", petal_length, "cm")
st.write("Petal width :", petal_width, "cm")


# --- Block 4: predict, and show the answer --------------------------------- #
# The model needs a 2D array with one row per flower, so we make one row.
# The numbers MUST be in the same order as in training.
input_row = np.array([[sepal_length, sepal_width, petal_length, petal_width]])

# Call the model. This is the f(x) step, the main job of the app.
predicted_number = model.predict(input_row)[0]
predicted_name = species_names[predicted_number]

# How sure the model is about the chosen species, as a percentage.
probabilities = model.predict_proba(input_row)[0]
confidence_percent = round(100 * probabilities[predicted_number], 1)

# Show the prediction in large text, with the confidence next to it.
st.subheader("Prediction")
st.metric(label="Predicted species", value=predicted_name)
st.write("Model confidence:", confidence_percent, "%")
'''

with open("app.py", "w") as app_file:
    app_file.write(app_source)

print("Wrote app.py to this folder:", len(app_source),
      "characters of Python.")

## Step 1.11 · Read the app that the notebook wrote

Let us print `app.py` and read it. It is about sixty lines. It has the same four
blocks that every model app has. You do not need to learn Streamlit by heart.
Only see the shape. In step 3 you read these blocks again and change them.

In [ ]:
# Show the app that we wrote, so you can read it without leaving the notebook.
with open("app.py") as app_file:
    print(app_file.read())

## The four blocks of every model app

In the printed script above you can find these four blocks, in this order.

1. **Load the saved model, only once.** Near the top, one function reads
   `iris_model.joblib`. `@st.cache_resource` keeps the model in memory, so the app
   does not load the file again on every click.
2. **Show the page heading.** `st.title(...)` and a short line of text.
3. **Collect the inputs.** There is one slider for each measurement. Each widget,
   like `st.sidebar.slider(...)`, is one line. It gives back the number that the
   user chose.
4. **Predict, and show the answer.** The four numbers go into one row, in the same
   order as in training. `model.predict(...)` is the `f(x)` call. The answer is
   shown in large text, with a confidence percentage, not as a bare label.

One more thing to know: when the user moves a slider, Streamlit runs `app.py`
again, from the first line to the last. That is why the page always shows the new
answer, and that is why we load the model only once.

The hard part, training the model, you did in earlier sessions. The app is only
these four blocks around the model.

## Step 1.12 · Write requirements.txt (the packages the cloud must install)

When you deploy, the cloud computer starts empty. It has none of your packages.
The cloud computer installs the packages in requirements.txt. This file gives the
exact name and version of each package. The versions matter: a model saved with
one version of scikit-learn may not load with a different version.

In [ ]:
# Write the requirements file next to the app, so the folder has every file
# you need. The cloud computer installs the packages in requirements.txt,
# at these exact versions.
requirements_text = '''# Packages the Streamlit app needs. The cloud computer starts empty.
# The cloud computer installs the packages in requirements.txt, at these exact
# versions. Use the same versions as your laptop, so the saved model file loads.
streamlit==1.40.1
scikit-learn==1.8.0
numpy==2.4.6
joblib==1.5.3
'''

with open("requirements.txt", "w") as req_file:
    req_file.write(requirements_text)

print("Wrote requirements.txt:")
print(requirements_text)

## Checkpoint 1 · Three new files in your folder

This notebook has now written three files in this folder:

- `iris_model.joblib`: the saved model file (a few KB)
- `app.py`: the web app (about 60 lines)
- `requirements.txt`: the list of packages, with version numbers

**Check your versions.** The cloud must install the same versions that you used to
save the model. Add one new cell at the end of this notebook and run it:

```python
import sklearn, numpy, joblib
print("scikit-learn ==", sklearn.__version__)
print("numpy        ==", numpy.__version__)
print("joblib       ==", joblib.__version__)
```

Open `requirements.txt`. If your numbers are different, change them to YOUR
numbers. Keep the `streamlit` line as it is. Colab users must do this, because
Colab often has different versions.

**Colab users:** click the folder icon on the left. Then download each of the three
files (⋮ → Download), and keep them in one folder on your laptop. Colab deletes the
files when the session ends.

Show your three files to your neighbour. Then go to step 2.

## Step 2 · Run the app on your laptop (in a terminal, not here)

Colab-only students cannot do this step. Watch with a neighbour, and join again at
step 3.

Open a **terminal** (in VS Code: Terminal → New Terminal). Then type these three
commands:

```
# 1. go to the folder that has app.py
cd sessions/S27_streamlit_deployment/notebooks

# 2. install Streamlit (only the first time)
uv pip install streamlit==1.40.1      # or:  pip install streamlit==1.40.1

# 3. start the app
streamlit run app.py
```

If the terminal says `streamlit` is not found, use `python -m streamlit run app.py`.

Streamlit starts a small web server. A browser tab opens by itself, usually at
`http://localhost:8501`. If no tab opens, copy the Local URL from the terminal into
your browser. "localhost" means "this computer". Nobody else can open this address
yet. Steps 4 and 5 change that.

Do not run `streamlit run` inside a notebook cell. The server never stops, so the
cell would never finish. Always use a terminal.

**Checkpoint 2 · The app works on your laptop**

- You see the page "Iris flower classifier" in your browser.
- Move a slider on the left. The prediction on the right changes. (Try petal
  length near 5 cm.)
- Keep the terminal open. If you close it, the app stops.
- To stop the app yourself, click in the terminal and press `Ctrl + C`.

## Step 3 · Read the app, then change it

Open `app.py` in VS Code (or any text editor) and find the four blocks described
above. Then make the app yours with three small changes:

```python
# 1. put your name in the title
st.title("Iris flower classifier, by Priya")

# 2. add one line under the title
st.caption("Made in S27, Applications of Mathematics in Industry")

# 3. change a starting value: petal length now starts at 4.5
petal_length = st.sidebar.slider("Petal length (cm)", 1.0, 7.0, 4.5)
```

Save the file. In the browser, click **Rerun** at the top right. Your change
appears. Colab-only students: make these changes in the downloaded `app.py` with
any text editor. You will see the result after step 5.

**Checkpoint 3:** your name is in the title of the app.

## Step 4 · Upload the files to a public GitHub repo

The cloud reads your app from a public GitHub repo. A repo (repository) is a
project folder on GitHub.

1. On github.com, click **+** (top right) → **New repository**.
2. Name: `iris-app`. Choose **Public**. Turn on **Add a README file**. Click
   **Create repository**.
3. On the new repo page, click **Add file** → **Upload files**.
4. Drag in `app.py`, `iris_model.joblib` and `requirements.txt`. Upload the three
   files together, not inside a folder.
5. Click the green **Commit changes** button at the bottom.

**Checkpoint 4:** your repo shows four files at the top level: `README.md`,
`app.py`, `iris_model.joblib` and `requirements.txt`. The word Public appears next
to the repo name.

## Step 5 · Deploy on Streamlit Community Cloud

It is free, and you need no credit card. You log in with your GitHub account.

1. Go to [share.streamlit.io](https://share.streamlit.io) and click **Continue
   with GitHub**.
2. When it asks, allow Streamlit to see your GitHub account.
3. Click **Create app** (top right).
4. Choose **Deploy a public app from GitHub**.
5. Fill in the form:
   - **Repository:** `yourname/iris-app`
   - **Branch:** `main`
   - **Main file path:** `app.py`
   - **App URL:** choose a short name, for example `priya-iris`
   - **Advanced settings:** choose the same Python version as your laptop (type
     `python --version` in the terminal) or Colab.
6. Click **Deploy**.

Button names on the site can change a little over time. If a button is not exactly
where this text says, look for the same words.

The build takes 2 to 5 minutes. The cloud computer starts empty. The cloud
computer installs the packages in requirements.txt. Then it runs `app.py`, in the
same way you did in step 2. Do not refresh the page again and again. Wait for the
app to appear.

**Checkpoint 5 · Open your link on your phone**

- Your app opens at a link like `https://your-name.streamlit.app`.
- Open the same link on your phone. Use mobile data, not college Wi-Fi.
- Move a slider on the phone. The answer changes.

Your model is now on the internet.

## Step 6 · Update the live app, and write the README

After the first deploy, you never deploy again. When you change a file in the repo,
the app changes by itself.

1. Open `app.py` in your repo on GitHub.
2. Click the pencil icon. Change the caption line.
3. Click **Commit changes**.
4. After about a minute, refresh your app. The change is live.

Then write a short README. Edit `README.md` on GitHub with the pencil icon, in the
same way:

- Line 1: the live link, so people can click it first.
- Line 2: what the app does, in one sentence.
- Line 3: how to use it: "Move the sliders on the left."
- Line 4: which model it uses, and its accuracy on test data.

**Checkpoint 6:** the change is on your live app, and the README starts with your
live link.

**Another free place to host the app: Hugging Face Spaces**

1. Create a new **Space** at [huggingface.co/spaces](https://huggingface.co/spaces)
   and choose **Streamlit** as the SDK.
2. Put the same files in the Space's repo.
3. The Space installs the packages in `requirements.txt` and gives you a public
   link.

**Things to watch**

- **Keep the versions fixed** in `requirements.txt` (this is done for you). A
  model saved with one version of scikit-learn may not load with another.
- **Use the file name alone** in the repo (`joblib.load("iris_model.joblib")`).
  Never use a laptop path like `C:/Users/me/...`. A laptop path works on your
  laptop, but it fails on the cloud.
- **Never put passwords or API keys in the repo.** A public repo is visible to
  everyone. Use the Secrets box in the Streamlit Cloud app settings.
- Free apps **go to sleep** when nobody uses them. The first visit after that
  takes 10 to 30 seconds to wake the app. This is fine for a demo, but tell people
  when you send them the link.

This is a demo app. It is right for a portfolio, a recruiter or your capstone. It
is not a full company system (it has no support for many users, and no
monitoring). In this course we use only Streamlit and its free hosting. We do not
use Docker or FastAPI.

## What you did

You trained a model, **saved it to a file**, and **loaded it back** to predict,
without training it again. Then this notebook **wrote the web app** that uses the
model, and you read its four blocks. In steps 2 to 6 you ran the app, changed it,
and deployed it to a free public link.

Notice what this notebook did, and what it did not do. It made three real files in
this folder (`iris_model.joblib`, `app.py`, `requirements.txt`). It ran from top to
bottom and never started a server. That is how the work is divided: the notebook
makes the app, and the terminal (or the free cloud) runs it.

The saved model file holds the function `f(x)`. A web app is that function with a
public address. That is what "a model as a function exposed as a service" means.